This is the .ipynb file for the Proof of Concept (PoC) of our project.

In [ ]:
NAMES = ['Sergey Dutt', 'Angela DiAndreth', 'Edward Lomobo-Becerra']

# summarize AI use on each cell
AI_Contribution = [
    ['Cell #', 'summary'],
    ['Cell 2', 'code created by gemini to ensure that code runs identically, regardless of workspace']
]

This code sets up file paths automatically. It connects Google Drive if you are using Google Colab, or uses your project's local folder if you are running Jupyter on your own computer.

In [ ]:
import sys, os

# Detect environment
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Google Colab specific environment setup
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/my-project'
else:
    # Local Linux environment setup
    PROJECT_DIR = os.path.abspath('.')

print(f"Working directory: {PROJECT_DIR}")

Working directory: /home/sergey-dutt/06623/project


# Problem Statement:
Our goal is to use the Fischer Information Matrix (FIM) and D-optimal experimental design to isolate a smaller, optimal set of data points to maximise parameter sensitivity and minimise parameter uncertainty for estimating kinetic parameters. This would be done because noisy datasets can be difficult to model due to parameter instability and unnecessary computational overhead. This is an interesting project to tackle because using FIM and D-optimal design allows us to create an effective model for a system using imperfect data.


# Summary of Fisher Information Matrix & D-Optimal Design

## Modeling Viewpoint of an Observation

Consider a basic first-order linear system:
$$
\frac{dx(t)}{dt} = -k \cdot x(t), \quad x(0) = x_0$$
The closed-form analytical solution to this system represents our deterministic trajectory function $g(t_i; \theta)$, where $\theta = [k, x_0]^T$:
$$g(t_i; \theta) = x_0 \cdot e^{-k t_i}$$

When measuring this physical process at time $t_i$, the recorded value $y(t_i)$ consists of the true deterministic state plus measurement error:
$$y(t_i) = g(t_i; \theta) + \epsilon_i$$
$g(t_i; \theta)$ is the deterministic solution to the differential equation model at time $t_i$. $\epsilon_i$ is Gaussian error associated with a measurement. Think of an observation $y(t_i)$ not as a scalar, but as a realization drawn from a probability distribution centered at the modeled value $g(t_i; \theta)$.

## Probability Distributions

The Gaussian integral with normalization condition guarantees that the total probability over the entire distribution equals 1:
$$\int_{-\infty}^{\infty} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\big(y_i - g(t_i; \theta)\big)^2}{2\sigma^2} \right) dy_i = 1$$
This leads to the probability density function for a single observation $y_i$:
$$f(y_i; \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{\big(y_i - g(t_i; \theta)\big)^2}{2\sigma^2} \right)$$
Applying the natural logarithm to the density function across $N$ independent observations yields the joint log-likelihood $\ell(\theta) = \log f(y; \theta)$:
$$\ell(\theta) = -\frac{N}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^N \Big(y_i - g(t_i; \theta)\Big)^2$$

## Information Preservation via Log Properties

Taking the natural logarithm is an information-preserving transformation because $\log(x)$ is strictly monotonic and bijective. The critical properties of the likelihood landscape are preserved completely:

Stationary Points (Gradients):
$$\nabla_\theta \ell(\theta) = \frac{1}{f(\theta)} \nabla_\theta f(\theta)$$
Since $f(\theta) > 0$, the critical points (where the gradient is zero) of $\ell(\theta)$ and $f(\theta)$ are identical:
$$\nabla_\theta \ell(\theta) = \mathbf{0} \iff \nabla_\theta f(\theta) = \mathbf{0}$$

Curvature (Hessian at the Peak):
At the Maximum Likelihood Estimate $\hat{\theta}$, where $\nabla_\theta f(\hat{\theta}) = \mathbf{0}$, the second-derivative Hessian matrix simplifies to:
$$\mathbf{H}_{\ell(\hat{\theta})} = \frac{1}{f(\hat{\theta})} \mathbf{H}_{f(\hat{\theta})}$$
Because the score gradient zero-crossings and peak curvature ratios are directly proportional between $\ell(\theta)$ and $f(\theta)$, no information about the parameter space is lost.

## Fisher Information Matrix

The Fisher Information Matrix $I(\theta)$ measures the amount of information that an observable random variable carries about an unknown parameter vector $\theta$:
$$I(\theta) = \mathbb{E}\left[ \Big(\nabla_\theta \ell(\theta)\Big) \Big(\nabla_\theta \ell(\theta)\Big)^T \right]$$
With a bit of algebra, the partial derivative of the joint log-likelihood $\ell(\theta)$ with respect to parameter component $\theta_j$ is given by:
$$\frac{\partial \ell(\theta)}{\partial \theta_j} = \frac{1}{\sigma^2} \sum_{i=1}^N \Big(y_i - g(t_i; \theta)\Big) \frac{\partial g(t_i; \theta)}{\partial \theta_j}$$
Let $e_i = y_i - g(t_i; \theta)$ define the error residual for measurement $i$, and let $J$ define the $N \times p$ sensitivity Jacobian matrix:
$$J = \begin{bmatrix}  \frac{\partial g(t_1; \theta)}{\partial \theta_1} & \dots & \frac{\partial g(t_1; \theta)}{\partial \theta_p} \\ \vdots & \ddots & \vdots \\ \frac{\partial g(t_N; \theta)}{\partial \theta_1} & \dots & \frac{\partial g(t_N; \theta)}{\partial \theta_p} \end{bmatrix}$$
Using $J$ and the residual vector $e = [e_1, e_2, \dots, e_N]^T$, the full score vector $S(\theta) = \nabla_\theta \ell(\theta)$ simplifies to compact matrix notation:
$$S(\theta) = \frac{1}{\sigma^2} J^T e$$
Substituting $S(\theta)$ back into the expectation definition:
$$I(\theta) = \mathbb{E}\left[ S(\theta) S(\theta)^T \right] = \frac{1}{\sigma^2} J^T J$$

## D-Optimal Design

Note: The following geometric overview is extremely simplified, with significant underlying functional analysis and measure theory omitted. Because D-optimal Design is the primary optimization problem of this project, the finer algorithmic and operational details will continue to be explored as implementation progresses.

When calibrating an unknown parameter vector $\theta$, we seek $\hat{\theta}$, an approximately normal distribution centered at the true value of $\theta$, with variance as the inverse of the information matrix:
$$\hat{\theta} \sim \mathcal{N}\left(\theta_{\text{true}}, \, I(\theta)^{-1}\right)$$
Within this parameter space of $\hat{\theta}$ is an uncertainty region:
$$\mathcal{E} = \left\{ \theta \in \mathbb{R}^p \;\middle\vert{}\; (\theta - \hat{\theta})^T I(\theta) (\theta - \hat{\theta}) \le c^2 \right\}$$
Because $I(\theta)$ is a symmetric positive semi-definite matrix, it can be decomposed via orthogonal diagonalization:
$$I(\theta) = V \Lambda V^T$$
$V$ is an orthogonal matrix whose columns are the orthonormal eigenvectors of $I(\theta)$. $\Lambda$ is a diagonal matrix containing the eigenvalues $\lambda_1, \lambda_2, \dots, \lambda_p$ of $I(\theta)$. The eigenvectors ($V$) of $I(\theta)$ define the principal axis directions of the parameter error space. The eigenvalues ($\Lambda$) bound the semi-axis lengths: $a_i = \frac{c}{\sqrt{\lambda_i}}$.

For any diagonalizable matrix $A = P D P^{-1}$, the determinant equals the product of its eigenvalues.
$$\det(A) = \det(P D P^{-1}) = \det(P) \cdot \det(D) \cdot \det(P^{-1}) = \det(D)$$
Thus $\det(I) = \det(\Lambda) = \prod_{i=1}^p \lambda_i$.
$$\text{Vol}(\mathcal{E}) \propto \frac{1}{\sqrt{\det(I(\theta))}}$$
The best parameter fit occurs when the uncertainty space is minimized, which leads to the dual optimization problem of maximizing $\det(I(\theta))$. This optimization problem is the focus of this project for finding an optimal subset $\mathcal{D}^*$ of a full dataset $\mathcal{D}$ that contains the maximum possible information about the original system. Calibration to the optimal subset $\mathcal{D}^*$ yields two core advantages: significant noise reduction by omitting low-sensitivity measurements, and substantial computational efficiency during optimization.

#Resources needed

Our current idea revolves around using the data from the Silbur et al paper, but (as discussed further in the "Risks" section) if this data is not able to work with FIM/solve_ivp, we have a backup plan of using data from a swinging pendulum experiment. Both of these data sets are discussed here, and both are cited in the citations section.

For the reaction network system, the differential model will be obtained from [1] whose datasets are found in [2]. The system involves two parallel reactions, whose kinetic parameters are unknown, consisting of the reaction of 1,2,4-triazole to give either a desired product or its regioisomer. The paper uses dynamic flow experiments to accelerate the development of a digital twin of the system. The data therein will be validated according to the methods of the paper to create a comparative baseline. The FIM and D-optimal experimental design framework will then be used on this data.

For the swinging pendulum system, the data is obtained from a prior project that Sergey worked on, whose modeling tasks did not use FIM. The dataset contains the angle of the pendulum with respect to rest position against time, starting at an initial angle of 40 degrees and measuring every tenth of a second. This dataset contains different thicknesses of string that supported the weight of the pendulum, as the goal of that project was to create a model that would account for variable air-resistance against the pendulum. For this project, we would just choose one thickness (probably the thickest one because it has the most stable measurements), as we are more interested in using FIM to create a data subset that best matches the information of the origin dataset.


#Feasibility and Scope
This project is the right size for the class project because it can be easily broken up into three chunks. One person would be able to work on creating the differential model/models, one person would work on the FIM part and creating an optimal data subset, and the third person would determine how we could measure the effectiveness of our model with a calibration test selection using some sort of statistical analysis. The smallest version of this project would be to effectively model a kinetic system using FIM, and verify its accuracy with statistical analysis. If time allows, we would want to try different models using our FIM code or to estimate more parameters using the data from Silber et al paper.


## Assumptions and Risks when using FIM

The reaction network system has two unknown parameters and an available dataset, making it a strong candidate for our D-optimal design project. However, we are unsure about how the system of differential equations would behave in a solver that we have used in class (like solve_ivp), or how to guess a good initial condition; it is clear that the Silber et al. paper used more complex solver methods. Should this system prove too complex for a solver that we understand and can call from a python library, we can fall back on the simpler, pendulum problem. In a previous project that Sergey worked on, RK45 and a single initial condition were used to solve for the differential models of the pendulum's motion, giving us good reason to believe that solve_ivp would work just fine.

### 1. Model Specificity vs. Measurement Noise

$$y(t_i) = \underbrace{g(t_i; \theta)}_{\text{Physics / Dynamics}} + \underbrace{\epsilon_i}_{\text{Measurement Error}}$$

The Fisher Information Matrix $I(\theta)$ operates entirely under the assumption that $\epsilon_i$ represents purely unmodeled, random noise. If the deterministic model $g(t_i; \theta)$ is missing physics, $\epsilon_i$ absorbs structural error, causing the FIM to fail.

### 2. Independence and No Autocorrelation
We assume that measurement errors across different sampling points are uncorrelated:

$$\text{Cov}(\epsilon_i, \epsilon_j) = 0 \quad \forall i \neq j$$

This asserts that error at time $t_i$ provides zero information about error at time $t_{i+1}$. If serial correlation exists, $I(\theta) = \frac{1}{\sigma^2} J^T J$ underestimates parameter uncertainty.

### 3. Constant Variance (Homoscedasticity)
Standard FIM assumes constant measurement variance $\sigma^2$ across all observations. If noise increases as the signal grows , we must introduce a diagonal weighting matrix $W$:

$$I(\theta) = J^T W J$$

where $W = \text{diag}\left(\frac{1}{\sigma_1^2}, \frac{1}{\sigma_2^2}, \dots, \frac{1}{\sigma_N^2}\right)$ scales each residual by its point-specific variance.

#Citations
[1] K. Silber, P. Sagmeister, C. Schiller, J. D. Williams, C. A. Hone, and C. O. Kappe, “Accelerating reaction modeling using dynamic flow experiments, part 2: development of a digital twin,” React. Chem. Eng., vol. 8, no. 11, pp. 2849–2855, Nov. 2023, doi: 10.1039/D3RE00244F

[2] P. Sagmeister et al., “Experimental Data for Dynamic Flow Reactions of 1,2,4-Triazole with Acrylonitrile”. Zenodo, Jul. 28, 2023. doi: 10.5281/zenodo.10849200.

[3] Aster, Richard C., Brian Borchers, and Clifford H. Thurber. Parameter estimation and inverse problems. Vol. 90. Amsterdam: Elsevier academic press, 2005.
